# Epic 3: Complete TSP Solver Suite Demo

This notebook demonstrates the complete implementation of **Epic 3: TSP Solver Implementation** for solving the Traveling Salesman Problem on Singapore's MRT/LRT network.

## Implemented Algorithms
- **US-301**: Nearest Neighbor Heuristic (Constructive)
- **US-302**: 2-Opt Local Search (Local Improvement)
- **US-303**: Simulated Annealing (Metaheuristic)
- **US-304**: Genetic Algorithm (Population-based Metaheuristic)

## Table of Contents
1. [Setup and Data Loading](#setup)
2. [US-301: Nearest Neighbor Heuristic](#us-301)
3. [US-302: 2-Opt Local Search](#us-302)
4. [US-303: Simulated Annealing](#us-303)
5. [US-304: Genetic Algorithm](#us-304)
6. [Algorithm Comparison](#comparison)
7. [Conclusions](#conclusions)

<a id='setup'></a>
## 1. Setup and Data Loading

First, let's import the necessary modules and load the Singapore MRT/LRT network graph.

**Note:** If you get import errors, restart your Jupyter kernel (Kernel → Restart & Clear Output) and re-run all cells.

In [ ]:
# Import required libraries
import sys
import os

# Add parent directory to path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import time
from pathlib import Path

# Import TSP solver modules
from src.graph.builder import load_default_graph
from src.solvers.nearest_neighbor import (
    nearest_neighbor_tsp,
    nearest_neighbor_multi_start,
    nearest_neighbor_with_2opt
)
from src.solvers.two_opt import improve_tour_2opt
from src.solvers.simulated_annealing import simulated_annealing_tsp
from src.solvers.genetic_algorithm import genetic_algorithm_tsp
from src.utils.tour import calculate_tour_cost, validate_tour

print("✓ All imports successful")

In [ ]:
# Load the Singapore MRT/LRT network as a complete graph for TSP
print("Loading Singapore MRT/LRT network...")
print("Note: Converting to complete graph where edge weights are shortest path distances.")
print("This is necessary for TSP algorithms to work on sparse metro networks.\n")
G = load_default_graph(complete=True)

print(f"\nNetwork Statistics:")
print(f"  Total stations: {G.number_of_nodes()}")
print(f"  Total connections: {G.number_of_edges()}")
print(f"  Network is connected: {nx.is_connected(G)}")
print(f"  Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")

# Store for later use
n_stations = G.number_of_nodes()

<a id='us-301'></a>
## 2. US-301: Nearest Neighbor Heuristic

The Nearest Neighbor algorithm is a **greedy constructive heuristic** that:
1. Starts at a given station
2. Repeatedly visits the nearest unvisited station
3. Returns to the starting station

**Time Complexity:** O(n²)  
**Space Complexity:** O(n)

In [ ]:
# Run Nearest Neighbor with multi-start
print("Running Nearest Neighbor (multi-start, n=10)...\n")

start_time = time.time()
nn_tour, nn_cost, best_start = nearest_neighbor_multi_start(G, num_starts=10)
nn_time = time.time() - start_time

print(f"Nearest Neighbor Results:")
print(f"  Best starting station: {best_start} - {G.nodes[best_start]['name']}")
print(f"  Tour length: {len(nn_tour)} stations")
print(f"  Total travel time: {nn_cost:.2f} minutes ({nn_cost/60:.2f} hours)")
print(f"  Computation time: {nn_time:.4f} seconds")
print(f"  Average time per station: {nn_cost/n_stations:.2f} minutes")

<a id='us-302'></a>
## 3. US-302: 2-Opt Local Search

The 2-Opt algorithm is a **local search improvement heuristic** that:
1. Takes an initial tour
2. Iteratively reverses segments to reduce tour cost
3. Continues until no improvement is found (local optimum)

**Time Complexity:** O(n² × iterations)  
**Space Complexity:** O(n)

In [ ]:
# Apply 2-Opt to the Nearest Neighbor solution
print("Applying 2-Opt optimization to NN solution...\n")

start_time = time.time()
two_opt_tour, original_cost, two_opt_cost = improve_tour_2opt(
    nn_tour, 
    G,
    max_iterations=None,  # Run until convergence
    improvement_threshold=0.001,
    verbose=False
)
two_opt_time = time.time() - start_time

improvement = original_cost - two_opt_cost
pct_improvement = (improvement / original_cost) * 100

print(f"2-Opt Results:")
print(f"  Original cost (NN): {original_cost:.2f} minutes")
print(f"  Optimized cost: {two_opt_cost:.2f} minutes")
print(f"  Improvement: {improvement:.2f} minutes ({pct_improvement:.2f}%)")
print(f"  Computation time: {two_opt_time:.4f} seconds")
print(f"  Average time per station: {two_opt_cost/n_stations:.2f} minutes")

<a id='us-303'></a>
## 4. US-303: Simulated Annealing

Simulated Annealing is a **probabilistic metaheuristic** that:
1. Starts with an initial solution (or random tour)
2. Generates random neighbors (2-opt swaps)
3. Accepts improving moves always
4. Accepts worsening moves with probability based on temperature
5. Gradually decreases temperature (cooling) to converge

**Key Feature:** Can escape local optima through probabilistic acceptance of worse solutions.

### 4.1 Simulated Annealing from Random Start

In [ ]:
# Run Simulated Annealing from random initial solution
print("Running Simulated Annealing (from random start)...\n")

start_time = time.time()
sa_tour, sa_cost, sa_history = simulated_annealing_tsp(
    G,
    initial_tour=None,  # Random start
    initial_temp=100.0,
    max_iterations=10000,
    cooling_schedule='exponential',
    random_seed=42,
    verbose=False
)
sa_time = time.time() - start_time

print(f"Simulated Annealing Results:")
print(f"  Best cost found: {sa_cost:.2f} minutes ({sa_cost/60:.2f} hours)")
print(f"  Computation time: {sa_time:.4f} seconds")
print(f"  Iterations: {len(sa_history)}")
print(f"  Average time per station: {sa_cost/n_stations:.2f} minutes")

### 4.2 Simulated Annealing from NN Solution

In [ ]:
# Run SA starting from the NN solution
print("Running Simulated Annealing (from NN solution)...\n")

start_time = time.time()
sa_nn_tour, sa_nn_cost, sa_nn_history = simulated_annealing_tsp(
    G,
    initial_tour=nn_tour,  # Start from NN solution
    initial_temp=50.0,      # Lower temp since starting from good solution
    max_iterations=10000,
    cooling_schedule='exponential',
    random_seed=42,
    verbose=False
)
sa_nn_time = time.time() - start_time

print(f"Simulated Annealing (warm start) Results:")
print(f"  Starting cost (NN): {nn_cost:.2f} minutes")
print(f"  Best cost found: {sa_nn_cost:.2f} minutes")
print(f"  Improvement: {nn_cost - sa_nn_cost:.2f} minutes ({((nn_cost - sa_nn_cost)/nn_cost)*100:.2f}%)")
print(f"  Computation time: {sa_nn_time:.4f} seconds")

### 4.3 Visualize SA Convergence

In [ ]:
# Plot SA convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Random start SA
ax1.plot(sa_history, linewidth=1, alpha=0.7)
ax1.axhline(y=min(sa_history), color='r', linestyle='--', label=f'Best: {min(sa_history):.2f} min', alpha=0.7)
ax1.set_xlabel('Iteration', fontsize=11)
ax1.set_ylabel('Tour Cost (minutes)', fontsize=11)
ax1.set_title('Simulated Annealing (Random Start)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Plot 2: NN start SA
ax2.plot(sa_nn_history, linewidth=1, alpha=0.7, color='orange')
ax2.axhline(y=min(sa_nn_history), color='r', linestyle='--', label=f'Best: {min(sa_nn_history):.2f} min', alpha=0.7)
ax2.set_xlabel('Iteration', fontsize=11)
ax2.set_ylabel('Tour Cost (minutes)', fontsize=11)
ax2.set_title('Simulated Annealing (NN Start)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"SA explores the solution space by accepting worse solutions with decreasing probability.")

<a id='us-304'></a>
## 5. US-304: Genetic Algorithm

Genetic Algorithm is a **population-based metaheuristic** inspired by natural evolution:
1. Maintains a population of candidate solutions (tours)
2. Selects parents using tournament selection
3. Creates offspring through crossover (Order Crossover)
4. Applies mutation (2-opt moves)
5. Preserves best individuals (elitism)

**Key Feature:** Explores multiple solutions simultaneously and combines good characteristics from different tours.

### 5.1 Run Genetic Algorithm

In [ ]:
# Run Genetic Algorithm
print("Running Genetic Algorithm...\n")

start_time = time.time()
ga_tour, ga_cost, ga_history = genetic_algorithm_tsp(
    G,
    population_size=100,
    generations=500,
    mutation_rate=0.2,
    crossover_type='order',
    mutation_type='reverse',
    elitism_count=2,
    random_seed=42,
    verbose=False
)
ga_time = time.time() - start_time

print(f"Genetic Algorithm Results:")
print(f"  Population size: 100")
print(f"  Generations: 500")
print(f"  Best cost found: {ga_cost:.2f} minutes ({ga_cost/60:.2f} hours)")
print(f"  Computation time: {ga_time:.4f} seconds")
print(f"  Average time per station: {ga_cost/n_stations:.2f} minutes")

### 5.2 Visualize GA Evolution

In [ ]:
# Plot GA evolution
plt.figure(figsize=(12, 6))
plt.plot(ga_history, linewidth=2, color='green')
plt.axhline(y=ga_history[0], color='r', linestyle='--', label=f'Initial best: {ga_history[0]:.2f} min', alpha=0.7)
plt.axhline(y=ga_history[-1], color='b', linestyle='--', label=f'Final best: {ga_history[-1]:.2f} min', alpha=0.7)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('Best Tour Cost (minutes)', fontsize=12)
plt.title('Genetic Algorithm Evolution', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

improvement = ga_history[0] - ga_history[-1]
print(f"\nImprovement: {improvement:.2f} minutes ({(improvement/ga_history[0])*100:.2f}%)")
print(f"GA steadily improves through selection and crossover over generations.")

<a id='comparison'></a>
## 6. Algorithm Comparison

Let's compare all four algorithms across multiple dimensions.

In [ ]:
# Summary table
print("\n" + "="*90)
print("COMPREHENSIVE ALGORITHM COMPARISON")
print("="*90)

print(f"\nNetwork: {n_stations} stations, {G.number_of_edges()} connections")
print(f"\n{'Algorithm':<35} {'Time (s)':<12} {'Cost (min)':<15} {'Quality':<15}")
print("-" * 90)

# Find best cost
best_cost = min(nn_cost, two_opt_cost, sa_cost, sa_nn_cost, ga_cost)

# Print results
def quality_score(cost):
    gap = ((cost - best_cost) / best_cost) * 100
    return f"+{gap:.2f}%" if gap > 0.01 else "BEST"

print(f"{'Nearest Neighbor (US-301)':<35} {nn_time:<12.4f} {nn_cost:<15.2f} {quality_score(nn_cost):<15}")
print(f"{'2-Opt on NN (US-302)':<35} {two_opt_time:<12.4f} {two_opt_cost:<15.2f} {quality_score(two_opt_cost):<15}")
print(f"{'Simulated Annealing - Random (US-303)':<35} {sa_time:<12.4f} {sa_cost:<15.2f} {quality_score(sa_cost):<15}")
print(f"{'Simulated Annealing - NN Start (US-303)':<35} {sa_nn_time:<12.4f} {sa_nn_cost:<15.2f} {quality_score(sa_nn_cost):<15}")
print(f"{'Genetic Algorithm (US-304)':<35} {ga_time:<12.4f} {ga_cost:<15.2f} {quality_score(ga_cost):<15}")

print(f"\n" + "="*90)
print(f"Best solution found: {best_cost:.2f} minutes ({best_cost/60:.2f} hours)")
print(f"This represents {best_cost/n_stations:.2f} minutes average per station.")
print("="*90)

### 6.1 Visual Comparison

In [ ]:
# Create comparison visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar chart - Cost comparison
algorithms = ['NN', '2-Opt', 'SA\n(Random)', 'SA\n(NN)', 'GA']
costs = [nn_cost, two_opt_cost, sa_cost, sa_nn_cost, ga_cost]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

bars = ax1.bar(algorithms, costs, color=colors, alpha=0.7, edgecolor='black')
ax1.axhline(y=best_cost, color='red', linestyle='--', linewidth=2, label=f'Best: {best_cost:.2f} min')
ax1.set_ylabel('Tour Cost (minutes)', fontsize=12, fontweight='bold')
ax1.set_title('Solution Quality Comparison', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, cost in zip(bars, costs):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{cost:.1f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Bar chart - Time comparison
times = [nn_time, two_opt_time, sa_time, sa_nn_time, ga_time]
bars2 = ax2.bar(algorithms, times, color=colors, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Computation Time (seconds)', fontsize=12, fontweight='bold')
ax2.set_title('Computation Time Comparison', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, t in zip(bars2, times):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{t:.2f}s',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### 6.2 Speed vs Quality Trade-off

In [ ]:
# Plot speed vs quality trade-off
plt.figure(figsize=(10, 7))

# Create scatter plot
plt.scatter(nn_time, nn_cost, s=200, c='#3498db', marker='o', label='Nearest Neighbor', edgecolors='black', linewidth=2, alpha=0.8)
plt.scatter(two_opt_time, two_opt_cost, s=200, c='#2ecc71', marker='s', label='2-Opt', edgecolors='black', linewidth=2, alpha=0.8)
plt.scatter(sa_time, sa_cost, s=200, c='#e74c3c', marker='^', label='SA (Random)', edgecolors='black', linewidth=2, alpha=0.8)
plt.scatter(sa_nn_time, sa_nn_cost, s=200, c='#f39c12', marker='D', label='SA (NN Start)', edgecolors='black', linewidth=2, alpha=0.8)
plt.scatter(ga_time, ga_cost, s=200, c='#9b59b6', marker='*', label='Genetic Algorithm', edgecolors='black', linewidth=2, alpha=0.8)

plt.xlabel('Computation Time (seconds)', fontsize=13, fontweight='bold')
plt.ylabel('Tour Cost (minutes)', fontsize=13, fontweight='bold')
plt.title('Algorithm Trade-off: Speed vs Quality', fontsize=15, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInsights:")
print("• Lower-left corner is ideal (fast + high quality)")
print("• Nearest Neighbor: Fastest but lowest quality")
print("• 2-Opt: Good balance of speed and quality")
print("• Metaheuristics (SA, GA): Better exploration but slower")

<a id='conclusions'></a>
## 7. Conclusions and Recommendations

### Algorithm Summary

#### US-301: Nearest Neighbor Heuristic
**Type:** Constructive Greedy Heuristic

**Strengths:**
- ✓ Extremely fast (< 1 second)
- ✓ Simple to understand and implement
- ✓ Deterministic results
- ✓ Good for generating initial solutions

**Limitations:**
- ✗ Solution quality varies with starting point
- ✗ No mechanism to escape poor early choices
- ✗ Typically 10-30% worse than optimal

**Best Use:** Initial solution generation, baseline comparison

---

#### US-302: 2-Opt Local Search
**Type:** Local Improvement Heuristic

**Strengths:**
- ✓ Guarantees improvement over initial solution
- ✓ Fast convergence
- ✓ Deterministic
- ✓ Simple and reliable

**Limitations:**
- ✗ Finds local optimum only
- ✗ Quality depends on initial solution
- ✗ Cannot escape local minima

**Best Use:** Refining solutions from constructive heuristics, post-processing

---

#### US-303: Simulated Annealing
**Type:** Probabilistic Metaheuristic

**Strengths:**
- ✓ Can escape local optima
- ✓ Balances exploration and exploitation
- ✓ Flexible cooling schedules
- ✓ Works well with good initial solutions

**Limitations:**
- ✗ Slower than local search
- ✗ Results vary between runs (stochastic)
- ✗ Requires parameter tuning (temperature, cooling)

**Best Use:** Medium-sized instances, when time permits, refining good solutions

---

#### US-304: Genetic Algorithm
**Type:** Population-based Evolutionary Metaheuristic

**Strengths:**
- ✓ Explores multiple solutions simultaneously
- ✓ Can combine good characteristics from different tours
- ✓ Robust across problem instances
- ✓ Natural parallelization potential

**Limitations:**
- ✗ Slowest algorithm (population overhead)
- ✗ Many parameters to tune
- ✗ Stochastic results
- ✗ No guarantee of improvement per generation

**Best Use:** Large instances, when computation time is not critical, research applications

---

### Recommended Workflows

**For Quick Solutions (< 5 seconds):**
```
Nearest Neighbor → 2-Opt
```
Best balance of speed and quality for practical applications.

**For High-Quality Solutions (< 30 seconds):**
```
Nearest Neighbor → 2-Opt → Simulated Annealing
```
Use SA to escape local optimum found by 2-Opt.

**For Research/Benchmarking:**
```
Run all algorithms → Compare → Ensemble best results
```
Use multiple approaches and select the best solution.

**For Large Instances (1000+ nodes):**
```
Nearest Neighbor → Genetic Algorithm (with limited generations)
```
GA's population-based approach scales better than SA.

---

### Performance on Singapore MRT/LRT Network

All algorithms successfully found valid TSP tours for the network.

**Key Findings:**
1. 2-Opt provides excellent quality improvement with minimal time cost
2. SA can further improve 2-Opt solutions given sufficient time
3. GA competitive with SA but requires more computation
4. Warm-starting metaheuristics (SA, GA) from NN improves results

---

### Future Enhancements

**Epic 4 - Visualization:**
- US-401: Interactive tour visualization on Singapore map
- US-402: Animation of algorithm convergence
- US-403: Comparison dashboards

**Epic 5 - Web Application:**
- US-501: Web interface for algorithm selection
- US-502: Real-time optimization progress
- US-503: Download and share results